# ⚡ 05 - Query Benchmarking & Time Travel

### 📌 Objective
In this notebook, we test the two most powerful operational features of Apache Iceberg:
1. **Partition Pruning:** Comparing query latency of a full table scan (12 monthly files) vs a single month query (1 file).
2. **Time Travel Auditing:** Querying historical point-in-time snapshots using `VERSION AS OF`.

In [ ]:
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Warmup Spark
spark.sql("SELECT count(*) FROM icelake.wikipedia.articles").collect()
print("✅ Spark initialized and warmed up!")

### Benchmark 1: Full Table Scan (12 Monthly Partitions)
Scans all ~54MB of data across all 12 monthly Parquet files.

In [ ]:
query_full = "SELECT count(*), avg(word_count_v2) FROM icelake.wikipedia.articles"
t0 = time.time()
res_full = spark.sql(query_full).collect()
t_full = time.time() - t0

print(f"⚡ Full Table Scan Execution Time: {t_full:.3f} seconds")
print(f"Result: {res_full}")

### Benchmark 2: Partition-Pruned Scan (Single Month - June 2024)
Iceberg reads manifest files and skips **11 out of 12 data files**, only scanning June 2024.

In [ ]:
query_pruned = """
    SELECT count(*), avg(word_count_v2) 
    FROM icelake.wikipedia.articles 
    WHERE created_date BETWEEN '2024-06-01' AND '2024-06-30'
"""
t0 = time.time()
res_pruned = spark.sql(query_pruned).collect()
t_pruned = time.time() - t0

print(f"🎯 Partition Pruned Scan Execution Time: {t_pruned:.3f} seconds")
print(f"Result: {res_pruned}")

speedup = (t_full / t_pruned) if t_pruned > 0 else 1.0
print(f"\n🚀 Speedup from Iceberg Partition Pruning: {speedup:.2f}x FASTER!")

### Demonstration 3: Time Travel Querying (`VERSION AS OF`)
Iceberg guarantees that snapshots are immutable. Let's find historical snapshots and query them.

In [ ]:
# List snapshots
snaps = spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary['total-records'] as total_records 
    FROM icelake.wikipedia.articles.snapshots 
    ORDER BY committed_at ASC
""").collect()

print("Available Snapshots:")
for s in snaps:
    print(f"  Snapshot ID: {s['snapshot_id']} | Records: {s['total_records']} | Date: {s['committed_at']}")

if len(snaps) >= 2:
    first_snap = snaps[0]['snapshot_id']
    latest_snap = snaps[-1]['snapshot_id']
    
    print(f"\n⏮️ Reading Snapshot #{first_snap} (Before updates):")
    spark.sql(f"SELECT count(*) as count_at_first_snapshot FROM icelake.wikipedia.articles VERSION AS OF {first_snap}").show()
    
    print(f"⏭️ Reading Latest Snapshot #{latest_snap} (Current state):")
    spark.sql(f"SELECT count(*) as current_count FROM icelake.wikipedia.articles VERSION AS OF {latest_snap}").show()